# Email Data Ingestion & Semantic Search Pipeline (CSV Version)

This Colab notebook demonstrates a complete enterprise-grade email ingestion pipeline using a CSV dataset.

In [ ]:
!pip install pandas numpy nltk sentence-transformers faiss-cpu beautifulsoup4

In [ ]:

import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
import faiss

nltk.download('punkt')


## Upload CSV File (Colab)

In [ ]:

from google.colab import files
uploaded = files.upload()


## Load CSV Dataset

In [ ]:

# Replace 'emails.csv' with your uploaded filename
df = pd.read_csv("emails.csv")
df.head()


## Cleaning & Normalization

In [ ]:

def clean_text(text):
    text = BeautifulSoup(str(text), "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Change 'body' to your actual text column name if needed
df = df.dropna(subset=['body'])
df['cleaned_body'] = df['body'].apply(clean_text)
df.head()


## Chunking

In [ ]:

def chunk_text(text, max_sentences=5):
    sentences = sent_tokenize(text)
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences])
        chunks.append(chunk)
    return chunks

chunks_data = []
for idx, row in df.iterrows():
    chunks = chunk_text(row["cleaned_body"])
    for c in chunks:
        chunks_data.append({
            "email_id": idx,
            "chunk_text": c
        })

chunk_df = pd.DataFrame(chunks_data)
chunk_df.head()


## Embeddings & FAISS Index

In [ ]:

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunk_df["chunk_text"].tolist(), show_progress_bar=True)

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(np.array(embeddings))

print("Vectors indexed:", index.ntotal)


## Semantic Search

In [ ]:

def semantic_search(query, top_k=5):
    q_emb = model.encode([query])
    distances, indices = index.search(np.array(q_emb), top_k)
    
    results = []
    for i in indices[0]:
        results.append(chunk_df.iloc[i]["chunk_text"])
    
    return results

query = "meeting schedule"
results = semantic_search(query)

for i, r in enumerate(results, 1):
    print(f"Result {i}:\n", r, "\n")
